In [23]:
import requests
import pandas as pd
from collections import defaultdict

#some setup required for hardcoded variables/ones that Sleeper API doesn't support at the moment
LEAGUE_ID = "1180249348364861440"
START_YEAR = 2020
CURRENT_YEAR = 2025

#seasons have changed length and there are non-fantasy weeks played that were getting returned
MAX_WEEK_PER_SEASON = {
    2020: 16,
    2021: 17,
    2022: 17,
    2023: 17,
    2024: 17,
    2025: 17
}

#map the managers' IDs to simple names
name_map = {
    "HaydenRoy": "Hayden",
    "JJFISCH": "Joe",
    "Jaredb11": "Jared",
    "JustinCasalini7": "Justin",
    "PapaWombat": "Joe",
    "PaulMarch": "Paul",
    "PaulMarch14": "Paul",
    "Tousi": "Dave",
    "Woodwoodwood": "Zach",
    "Zimm3r": "Zimmer",
    "awoody9": "Alex",
    "bartzy16": "Chris",
    "josephfischbach": "Joe",
    "papawombat": "Joe",
    "sambudin": "Sam",
    "sbtwelve": "Sam",
    "yungnito": "Bennett"
}

#couldn't extract league IDs from other endpoint, will check into later
season_league_ids = {
    2020: "566127140426665984",
    2021: "650525115267760128",
    2022: "784545497384931328",
    2023: "917523226949967872",
    2024: "1048832409841676288",
    2025: LEAGUE_ID
}

In [24]:
def get_matchups(league_id, week):
    url = f"https://api.sleeper.app/v1/league/{league_id}/matchups/{week}"
    r = requests.get(url)
    return r.json() if r.status_code == 200 else []

def get_rosters(league_id):
    url = f"https://api.sleeper.app/v1/league/{league_id}/rosters"
    r = requests.get(url)
    if r.status_code != 200:
        return {}
    return {r["roster_id"]: r["owner_id"] for r in r.json()}

def get_users(league_id):
    url = f"https://api.sleeper.app/v1/league/{league_id}/users"
    r = requests.get(url)
    if r.status_code != 200:
        return {}
    return {u["user_id"]: u["display_name"] for u in r.json()}

def get_player_map():
    url = "https://api.sleeper.app/v1/players/nfl"
    players = requests.get(url).json()
    return {
        pid: {
            "name": p.get("full_name") or f"{p.get('first_name','')} {p.get('last_name','')}",
            "position": p.get("position", "UNK")
        }
        for pid, p in players.items()
    }



In [25]:
all_stats = defaultdict(lambda: defaultdict(float))
games_played = defaultdict(lambda: defaultdict(int))

for year in range(START_YEAR, CURRENT_YEAR + 1):
    print(f"Pulling {year} season data")
    season_league_id = season_league_ids.get(year)
    if not season_league_id:
        print(f"Missing league ID for {year}")
        continue
    
    rosters = get_rosters(season_league_id)
    users = get_users(season_league_id)
    
    max_week = MAX_WEEK_PER_SEASON.get(year, 17)
    
    for week in range(1, max_week + 1):
        matchups = get_matchups(season_league_id, week)
        for team in matchups:
            owner_id = rosters.get(team["roster_id"])
            owner_name = users.get(owner_id, "Unknown Owner")
            owner_name = name_map.get(owner_name, owner_name)
            
            # Only count starters
            starters = set(team.get("starters", []))
            for player_id in starters:
                pts = team.get("players_points", {}).get(player_id, 0)
                all_stats[owner_name][player_id] += pts
                games_played[owner_name][player_id] += 1







Pulling 2020 season data
Pulling 2021 season data
Pulling 2022 season data
Pulling 2023 season data
Pulling 2024 season data
Pulling 2025 season data


In [26]:
player_map = get_player_map()

results = []
for owner, players in all_stats.items():
    for pid, pts in players.items():
        info = player_map.get(pid, {"name": pid, "position": "UNK"})
        results.append([
            owner,
            info["name"],
            info["position"],
            pts,
            games_played[owner][pid]
        ])

df = pd.DataFrame(results, columns=["Manager", "Player", "Position", "Total Points", "Games Started"])
df.head()



,Manager,Player,Position,Total Points,Games Started
0,Zach,Diontae Johnson,WR,261.00,22
1,Zach,Baker Mayfield,QB,56.66,4
2,Zach,Jordan Howard,RB,6.70,1
3,Zach,A.J. Brown,WR,6.40,1
4,Zach,Darren Waller,TE,204.60,22


In [27]:
lineups = []

for manager, group in df.groupby("Manager"):
    lineup = {"Manager": manager}
    
    # QB
    qb = group[group["Position"]=="QB"].sort_values("Total Points", ascending=False).head(1)
    lineup["QB"] = qb.iloc[0]["Player"] if not qb.empty else None
    lineup["QB Points"] = qb.iloc[0]["Total Points"] if not qb.empty else 0
    lineup["QB Games Started"] = qb.iloc[0]["Games Started"] if not qb.empty else 0
    
    # RB1, RB2
    rbs = group[group["Position"]=="RB"].sort_values("Total Points", ascending=False).head(2)
    for i in range(2):
        lineup[f"RB{i+1}"] = rbs.iloc[i]["Player"] if len(rbs) > i else None
        lineup[f"RB{i+1} Points"] = rbs.iloc[i]["Total Points"] if len(rbs) > i else 0
        lineup[f"RB{i+1} Games Started"] = rbs.iloc[i]["Games Started"] if len(rbs) > i else 0
    
    # WR1, WR2
    wrs = group[group["Position"]=="WR"].sort_values("Total Points", ascending=False).head(2)
    for i in range(2):
        lineup[f"WR{i+1}"] = wrs.iloc[i]["Player"] if len(wrs) > i else None
        lineup[f"WR{i+1} Points"] = wrs.iloc[i]["Total Points"] if len(wrs) > i else 0
        lineup[f"WR{i+1} Games Started"] = wrs.iloc[i]["Games Started"] if len(wrs) > i else 0
    
    # TE
    te = group[group["Position"]=="TE"].sort_values("Total Points", ascending=False).head(1)
    lineup["TE"] = te.iloc[0]["Player"] if not te.empty else None
    lineup["TE Points"] = te.iloc[0]["Total Points"] if not te.empty else 0
    lineup["TE Games Started"] = te.iloc[0]["Games Started"] if not te.empty else 0
    
    # FLEX (best remaining RB/WR/TE)
    flex_pool = group[group["Position"].isin(["RB","WR","TE"])].sort_values("Total Points", ascending=False)
    already_used = {
        lineup.get("RB1"), lineup.get("RB2"),
        lineup.get("WR1"), lineup.get("WR2"),
        lineup.get("TE")
    }
    flex_pick = flex_pool[~flex_pool["Player"].isin(already_used)].head(1)
    lineup["FLEX"] = flex_pick.iloc[0]["Player"] if not flex_pick.empty else None
    lineup["FLEX Points"] = flex_pick.iloc[0]["Total Points"] if not flex_pick.empty else 0
    lineup["FLEX Games Started"] = flex_pick.iloc[0]["Games Started"] if not flex_pick.empty else 0
    
    # Total points and games for the lineup
    lineup["Total Points"] = sum([
        lineup.get("QB Points",0),
        lineup.get("RB1 Points",0),
        lineup.get("RB2 Points",0),
        lineup.get("WR1 Points",0),
        lineup.get("WR2 Points",0),
        lineup.get("TE Points",0),
        lineup.get("FLEX Points",0)
    ])
    
    lineup["Total Games Started"] = sum([
        lineup.get("QB Games Started",0),
        lineup.get("RB1 Games Started",0),
        lineup.get("RB2 Games Started",0),
        lineup.get("WR1 Games Started",0),
        lineup.get("WR2 Games Started",0),
        lineup.get("TE Games Started",0),
        lineup.get("FLEX Games Started",0)
    ])
    
    lineups.append(lineup)

lineups_df = pd.DataFrame(lineups)
lineups_df



,Manager,QB,QB Points,QB Games Started,RB1,RB1 Points,RB1 Games Started,RB2,RB2 Points,RB2 Games Started,...,WR2 Points,WR2 Games Started,TE,TE Points,TE Games Started,FLEX,FLEX Points,FLEX Games Started,Total Points,Total Games Started
0,Alex,Jared Goff,618.54,32,Aaron Jones,805.50,61,Miles Sanders,479.40,49,...,377.30,36,David Njoku,429.50,38,Josh Jacobs,260.70,15,3770.24,299
1,Bennett,Kyler Murray,1311.80,68,D'Andre Swift,321.30,25,David Montgomery,273.82,28,...,365.30,24,Hunter Henry,429.90,50,Courtland Sutton,330.40,29,3595.12,252
2,Chris,Matthew Stafford,422.16,22,Josh Jacobs,467.50,30,Dalvin Cook,318.80,14,...,218.80,18,Sam LaPorta,206.60,18,Alvin Kamara,302.50,24,2263.76,152
3,Dave,Tua Tagovailoa,990.70,61,Jonathan Taylor,1316.80,77,Saquon Barkley,1123.90,74,...,1022.68,67,T.J. Hockenson,770.80,73,Tyler Lockett,607.40,52,7108.22,489
4,Hayden,Jared Goff,926.72,55,Tony Pollard,508.90,40,Devin Singletary,312.20,32,...,499.40,33,Tucker Kraft,217.10,19,Jerry Jeudy,425.90,44,3456.52,275
5,Jared,Justin Herbert,1721.72,89,Bijan Robinson,807.10,48,Travis Etienne,645.10,51,...,506.20,42,George Kittle,917.40,67,Ezekiel Elliott,552.46,44,5827.18,382
6,Joe,Josh Allen,533.62,22,David Montgomery,443.30,29,Joe Mixon,353.90,26,...,360.30,22,Kyle Pitts,371.80,46,James Conner,306.80,23,2831.62,204
7,Justin,Josh Allen,769.70,30,Jahmyr Gibbs,813.10,46,Kyren Williams,685.90,46,...,473.70,35,Jake Ferguson,304.00,28,De'Von Achane,554.20,33,4127.30,249
8,Paul,Patrick Mahomes,2035.70,93,Derrick Henry,1114.62,62,Leonard Fournette,464.00,36,...,515.68,57,Travis Kelce,974.46,65,DeAndre Hopkins,395.40,40,6383.06,419
9,Sam,Sam Darnold,647.14,38,Christian McCaffrey,849.16,42,James Cook,441.20,25,...,490.10,37,Evan Engram,488.70,53,Javonte Williams,384.80,37,3862.40,265


In [30]:
# Save full player history
df.to_csv(r"C:\Users\zachr\Desktop\sleeper api demo/all_time_players_by_team.csv", index=False)

# Save all-time top lineup per manager
lineups_df.to_csv(r"C:\Users\zachr\Desktop\sleeper api demo/all_time_lineups.csv", index=False)

print("all_time_players_by_team.csv downloaded successfully")
print("all_time_lineups.csv downloaded successfully")


all_time_players_by_team.csv downloaded successfully
all_time_lineups.csv downloaded successfully
